# RNN, LSTM, GRU — Notes

## Section 1 — RNN (Recurrent Neural Network)

**Note 1 — What it is**
An RNN is a neural network built for sequential data (text, speech, time series). Unlike a normal neural net, it has memory — it carries forward information from earlier words/tokens to influence what it predicts now.

**Note 2 — How it works**
At every time step, it takes two things as input: the current word, and the hidden state (memory) from the previous step. It combines them to produce a new hidden state and an output:

```
New Hidden State = f(Current Word + Previous Hidden State)
```

So the output at any point depends on both the current word and everything the model has retained from earlier.

**Note 3 — Worked example: "FD amount not credited, my FD matured last week."**
Meaning: the customer is saying the FD matured, but the amount still hasn't been credited.

Why RNN gets this right:
- It remembers "not credited" even after later seeing "matured last week."
- It connects the two ideas — FD maturity (event completed) + amount not credited (problem) — into a single negative complaint, not two neutral facts.
- It understands the payout delay *after* maturity, not before.

**Note 4 — Why a plain (non-recurrent) neural network fails here**
- It sees the sentence with no order or memory — words are just a bag.
- It may latch onto "FD matured last week" and call it neutral/positive.
- It misses that "not credited" is what flips the whole meaning negative.
- Result: misclassifies a genuine complaint as "FD Matured" (non-issue) instead of "Complaint."

**Note 5 — Key strength of RNNs**
They're good at sequence understanding because they connect information across time — carrying earlier context forward while reading later parts of the sentence.

---

## Section 2 — Limitations of Simple RNN

**Note 6 — Problem A: Short-term memory (forgetting)**
In a long email, by the time the RNN reaches the end, it's forgotten the beginning.

Worked example: an email clearly says early on that an FD should *not* be auto-renewed (nominee details pending), but after several paragraphs about branch visits, KYC, and account updates, it ends with "FD was renewed successfully." By the time the RNN reaches that final line, it has already forgotten the earlier "not" — so it may wrongly predict "customer wanted renewal," when the customer actually did **not** want it. Cause: RNN processes step-by-step, and early information weakens/decays the further the sequence goes.

**Note 7 — Problem B: Vanishing gradient**
During training (backpropagation), gradients shrink as they travel back through many time steps. Once they get small enough, weight updates basically stop happening for distant connections — so the model can't learn long-range dependencies. In the same example: the relationship between the early "not" and the later "renewed" needs a gradient strong enough to travel that whole distance — but it decays away, so the model never properly learns that connection.

**Note 8 — Short-term memory vs vanishing gradient: related, but NOT identical**
- **What it is:** the short-term memory problem is the *symptom* — what we observe in the output. The vanishing gradient problem is the *technical cause*, happening during training.
- **When it happens:** short-term memory loss shows up during prediction. Vanishing gradient shows up during backpropagation.
- **In the same example:** short-term memory — the RNN forgets "not" by the time it reaches "renewed." Vanishing gradient — the gradient connecting "not" and "renewed" shrinks so much during training that the weights never properly learn that link.
- **Effect:** short-term memory loss means the model loses long-term context at prediction time. Vanishing gradient means the model can't learn long-term relationships in the first place.
- **Bottom line:** the vanishing gradient problem (a training-time cause) is *why* the short-term memory problem (an inference-time symptom) happens.

---

## Section 3 — LSTM (Long Short-Term Memory)

**Note 9 — What it is and its components**
LSTM is an advanced RNN that fixes the forgetting problem using "gates" that control what to remember, update, or forget — plus a separate long-term memory called the **Cell State**.

- **Cell State (Ct):** long-term memory carrying important info across the whole sequence.
- **Hidden State (Ht):** the current output / short-term memory.
- **Forget Gate:** decides what old information to throw away.
- **Input Gate:** decides what new information to store.
- **Output Gate:** decides what to actually output.

**Note 10 — Same auto-renew example, walked through gate by gate**
Sentence: "FD should not be auto-renewed ... FD was renewed successfully."

- **Forget Gate:** drops unimportant details (branch visit, KYC, account update chatter) but keeps the important context — "FD should not be auto-renewed."
- **Input Gate:** stores the key new fact — "not auto-renewed."
- **Cell State:** carries that "not renewed" fact across the entire long sequence without losing it.
- **Output Gate:** uses that preserved memory to generate the final, correct understanding — even after a long email, it still remembers the customer did *not* want renewal.

**Note 11 — Why LSTM is better than plain RNN**
- Keeps relevant memory across longer sentences.
- Handles context changes over time.
- Preserves long-term context properly.
- Solves the vanishing gradient problem.

---

## Section 4 — Bidirectional LSTM (BiLSTM)

**Note 12 — What it is**
A BiLSTM reads the text in both directions — left→right *and* right→left — instead of just one direction like a normal LSTM. It then combines both passes for a fuller understanding.

**Note 13 — Same auto-renew example, walked through**
- **Forward LSTM (left→right):** learns from earlier context → "FD should not be auto-renewed."
- **Backward LSTM (right→left):** learns from later context → "renewed successfully."
- **Combine both:** builds a better understanding of how "not" and "renewed" relate to each other.
- **Result:** the model correctly understands the customer did **not** want the renewal — by checking both directions instead of relying on one direction not to forget.

**Note 14 — Advantages vs. limitation**
- Advantages: better contextual understanding, captures both past *and* future dependencies, better NLP performance.
- Limitation: slower and more computationally expensive than a normal (one-directional) LSTM.

---

## Section 5 — GRU (Gated Recurrent Unit)

**Note 15 — What it is and its gates**
GRU is a simplified alternative to LSTM that solves the same short-term-memory and vanishing-gradient problems, but with fewer gates:
- **Update Gate (z):** decides how much old information to keep. Example: keeps important context → "FD should not be auto-renewed."
- **Reset Gate (r):** decides how much old information to forget. Example: discards less-important detail like branch visit, account update, interest rate discussion.

**Note 16 — Same auto-renew example, walked through**
- **Problem in simple RNN:** when the model reaches "renewed successfully," it may forget the earlier "not" and wrongly conclude the customer wanted renewal.
- **GRU result:** even after a long sequence, GRU still remembers "not auto-renewed."
- **Final understanding:** the model correctly concludes the customer did *not* want renewal.

**Note 17 — Advantages vs. limitations**
- Advantages: faster and simpler than LSTM, still handles long-term dependencies well, requires fewer parameters.
- Limitation: slightly less expressive than LSTM for very complex sequences.

---

## Section 6 — RNN vs LSTM vs GRU, side by side

**Note 18 — Feature-by-feature comparison**
- **Memory:** RNN = short · LSTM = long · GRU = long
- **Vanishing gradient:** RNN = severe · LSTM = solved · GRU = solved
- **Long-context handling:** RNN = poor · LSTM = excellent · GRU = very good
- **Gates:** RNN = none · LSTM = 3 gates · GRU = 2 gates
- **Cell state:** RNN = no · LSTM = yes · GRU = no
- **Parameters:** RNN = few · LSTM = most · GRU = moderate
- **Training speed:** RNN = fast · LSTM = slow · GRU = faster
- **Complexity:** RNN = simple · LSTM = complex · GRU = moderate
- **Accuracy (NLP tasks):** RNN = lowest · LSTM = highest · GRU = close to LSTM
- **Best used for:** RNN = short sequences/small datasets · LSTM = long sequences with complex relationships · GRU = long sequences needing faster training

**Note 19 — When to use which**
- **Simple RNN:** short sentences, small context, simple problems.
- **LSTM:** long sentences, complex context, high accuracy required.
- **GRU:** long sentences, faster training needed, good performance trade-off between speed and accuracy.

---

## Section 7 — Key hyperparameters across RNN/LSTM/GRU

**Note 20 — Architecture-size hyperparameters**
- **Hidden Size (Hidden Units):** controls memory capacity. More units = richer patterns, but more compute/overfitting risk. Typical: 32, 64, 128, 256.
- **Number of Layers:** how many RNN/LSTM/GRU layers are stacked. More layers = deeper patterns from text. Typical: 1, 2, 3, 4.
- **Embedding Dimension:** size of each word's input vector. Higher = richer semantic meaning, but more compute. Typical: 50, 100, 200, 300 (this is exactly the `vector_size` parameter from Word2Vec).

**Note 21 — Data-shape hyperparameters**
- **Sequence Length (Time Steps):** how many words/tokens are processed per input. Longer = more context, but more computation. Typical: 16, 32, 64, 128.
- **Batch Size:** how many samples are processed together per training step. Larger batch = faster training but needs more memory. Typical: 32, 64, 100, 200.
- **Vocabulary Size:** number of unique words/subwords the model knows. Larger = better coverage, more memory. Typical: 5K, 10K, 20K, 50K, 100K+.

**Note 22 — Training-process hyperparameters**
- **Learning Rate:** controls how fast weights update. Too high → unstable training; too low → slow convergence. Typical: 0.1, 0.01, 0.001, 0.0001.
- **Epochs:** how many full passes over the dataset. More epochs = better learning until it converges, but too many = overfitting. Typical: 5, 10, 20, 50, 100.
- **Dropout:** randomly drops neurons during training to reduce overfitting. Typical: 0.1, 0.2, 0.3, 0.5.
- **Optimizer:** algorithm used to update weights — affects speed/stability of convergence. Typical: Adam (default), RMSProp, SGD.
- **Bidirectional:** whether the model reads sequences in both directions (BiLSTM-style) for better context. Typical: Yes / No.

---

**Quick takeaway:** RNN is the base model with short memory. LSTM adds gates + a cell state to preserve long-term information. GRU simplifies LSTM (fewer gates) and trains faster while still handling long dependencies well.